# Day 09 - Nested JSON and Complex Types

**Topic:** Nested JSON and Complex Types  
**Main environment:** Microsoft Fabric Notebook + Fabric Lakehouse  
**Focus:** ฝึกอ่าน nested JSON และแปลง complex types เช่น `struct`, `array`, `map` ให้เป็น tabular DataFrame ที่ใช้งานต่อได้

วันนี้เราจะเรียนวิธีทำงานกับข้อมูลกึ่งโครงสร้างแบบ nested JSON ใน PySpark โดยเน้น pattern ที่เจอบ่อยจาก API, event log และ application payload เช่น nested object, list ของ items, key-value attributes และ JSON string ที่ต้อง parse เพิ่ม ก่อน flatten ออกมาเป็น table สำหรับ Lakehouse


## Goal of the day

1. เข้าใจความต่างระหว่าง **StructType**, **ArrayType** และ **MapType** ใน PySpark
2. อ่าน nested JSON ด้วย explicit schema ได้
3. เข้าถึง nested fields ด้วย dot notation และ `F.col()` ได้
4. ใช้ `explode` / `explode_outer` เพื่อแตก array records ได้
5. Flatten nested JSON ให้กลายเป็น tabular DataFrame ที่นำไปใช้ต่อใน Lakehouse ได้


## Why it matters in production

ใน production pipeline ข้อมูลจาก API, event tracking, mobile app, web log หรือ partner integration มักไม่ได้มาเป็น table แบน ๆ ตั้งแต่แรก แต่จะมาเป็น JSON ที่มีหลายชั้น เช่น customer object, device object, list ของสินค้า และ dynamic attributes

ถ้า handle nested JSON ไม่ดี อาจเกิดปัญหาได้ เช่น:

- schema เปลี่ยนแล้ว pipeline อ่านผิดหรือ column กลายเป็น null โดยไม่รู้ตัว
- flatten แล้ว grain ของข้อมูลเปลี่ยน เช่น 1 event มีหลาย items พอ `explode` แล้ว 1 event กลายเป็นหลาย item rows
- explode array แล้ว records หาย เช่น บาง event ไม่มี field `items` หรือ `items` เป็น empty array ถ้าใช้ `explode(items)` event นั้นจะไม่ถูกสร้างเป็น row ในผลลัพธ์ ทำให้จำนวน event ลดลงโดยไม่รู้ตัว
- dynamic key ไม่มีจริง เช่น เรียก `attributes["campaign"]` แต่บาง event ไม่มี key นี้ ทำให้ได้ค่า null แทน
- raw payload ถูก drop เร็วเกินไป ทำให้ debug source issue ยาก

Production takeaway:

> Nested JSON ควรเริ่มจากเข้าใจ schema และ grain ของข้อมูลก่อน flatten โดยเฉพาะตอนใช้ explode กับ array เพราะจำนวน rows อาจเปลี่ยนได้


## Key concepts

| Concept | Meaning |
|---|---|
| StructType | Nested object หนึ่งชุดที่มี field ย่อย เช่น customer มี customer_id, name |
| ArrayType | List ของค่า หรือ list ของ nested object เช่น items ที่หนึ่ง event มีหลาย item records |
| MapType | Key-value field ที่ key อาจแตกต่างกันในแต่ละ record เช่น `attributes["campaign"]`, `attributes["channel"]` |
| Nested column access | การเข้าถึง field ชั้นในด้วย dot notation เช่น `F.col("customer.name")` |
| `explode` | แตก array ให้กลายเป็นหลาย rows แต่ถ้า array ว่างหรือ null จะไม่มี row ออกมา |
| `explode_outer` | แตก array เหมือน `explode` แต่ถ้า array ว่างหรือ null จะยังคง row ต้นทางไว้ โดย field ที่แตกจาก array จะเป็น `null` |
| `from_json` | Parse JSON string column ให้กลายเป็น struct ตาม schema ที่กำหนด |
| Flattening | การแปลง nested fields ให้กลายเป็น columns เช่น `customer.customer_id` → `customer_id` |
| Grain | ระดับความละเอียดของ row เช่น event-level หรือ item-level |


## Concept explanation

### Nested JSON คืออะไร?

Nested JSON คือ JSON ที่ไม่ได้มีแค่ key-value ชั้นเดียว แต่มี object หรือ array ซ้อนอยู่ข้างใน เช่น:

```json
{
  "event_id": "EVT001",
  "customer": {
    "customer_id": "C001",
    "name": "Alice"
  },
  "items": [
    {"product_id": "P001", "quantity": 1},
    {"product_id": "P002", "quantity": 2}
  ]
}
```

ใน Spark schema จะมองได้ประมาณนี้:

- `customer` เป็น `StructType` เพราะเป็น nested object ที่มี field ย่อย เช่น customer_id, name
- `items` เป็น `ArrayType` ของ `StructType` เพราะเป็น list ของ nested objects และแต่ละ item มี field ย่อย เช่น `product_id`, `quantity`
- ถ้ามี field แบบ dynamic key-value เช่น `attributes` ที่ key อาจเปลี่ยนไปตาม record มักเหมาะกับการ model เป็น `MapType`

### Struct: nested object

`struct` คือ column ที่ข้างในมี field ย่อย เช่น:

```python
F.col("customer.customer_id")
F.col("customer.name")
```

เวลา flatten เรามักดึง field ย่อยออกมาเป็น column แยก เช่น `customer_id`, `customer_name`

### Array: list of values หรือ list of objects

`array` คือ field ที่เก็บหลายค่าในหนึ่ง row เช่น order event หนึ่งรายการมีหลาย products

ถ้าต้องการวิเคราะห์ระดับ item-level ต้องใช้ `explode` หรือ `explode_outer` เพื่อเปลี่ยนจาก:

```text
1 event row with 3 items
```

เป็น:

```text
3 item rows
```

นี่คือจุดที่ต้องระวังมาก เพราะ **grain ของ DataFrame จะเปลี่ยนทันที**

### Map: dynamic key-value attributes

`map` เหมาะกับ attributes ที่ key อาจไม่ตายตัว เช่น:

```python
F.element_at("attributes", "campaign")
F.element_at("attributes", "channel")
```

ใน production ถ้า key สำคัญมาก ควร extract ออกมาเป็น column ชัดเจน เช่น `campaign`, `channel` เพื่อให้ง่ายต่อ validation และ downstream use

### JSON string column

บาง source ส่ง field หนึ่งมาเป็น string ที่ข้างในเป็น JSON อีกที เช่น:

```text
"{\"risk_score\": 0.82, \"payment_verified\": false}"
```

กรณีนี้ใช้ `from_json` เพื่อ parse string ให้กลายเป็น struct ก่อน แล้วค่อย select field ชั้นใน


## Mermaid diagram: Nested JSON Flattening Flow

```mermaid
flowchart LR
    A[Raw Nested JSON] --> B[Read with Explicit Schema]
    B --> C[Access Struct Fields]
    B --> D[Extract Map Keys]
    B --> E[Parse JSON String with from_json]
    B --> F[Explode Array]
    F --> G[Item-level Rows]
    C --> H[Flattened Event Columns]
    D --> H
    E --> H
    G --> I[Flattened Lakehouse Table]
    H --> I
```

Key idea:

> ต้องรู้ก่อนว่า output ที่ต้องการเป็น event-level หรือ item-level เพราะ `explode` จะเปลี่ยนจำนวน rows และ grain ของข้อมูล


## Setup / imports


In [1]:
import json

from pyspark.sql import functions as F
from pyspark.sql import types as T

StatementMeta(, b565ef77-0fec-4050-a2b7-11d6c8839678, 3, Finished, Available, Finished, False)

## Create mock data

ใน notebook นี้จะสร้าง mock nested JSON event data แล้วเขียนเป็น JSON Lines file ลง Lakehouse `Files/` เพื่อจำลองการอ่านข้อมูลจาก API/event source


In [2]:
# Python list of dicts; each dict represents one nested API event record
api_events = [
    {
        "event_id": "EVT001",
        "event_time": "2026-05-01T09:15:00Z",
        "source_system": "mobile_app",
        "customer": {
            "customer_id": "C001",
            "name": "Alice Wong",
            "segment": "gold",
        },
        "device": {
            "platform": "ios",
            "app_version": "2.4.1",
            "device_id": "IOS-001",
        },
        "location": {
            "country": "TH",
            "city": "Bangkok",
        },
        "items": [
            {"product_id": "P001", "product_name": "Wireless Mouse", "quantity": 1, "price": 890.0},
            {"product_id": "P002", "product_name": "USB-C Cable", "quantity": 2, "price": 250.0},
        ],
        "attributes": {
            "campaign": "may_promo",
            "channel": "push_notification",
            "coupon": "MAY100",
        },
        "event_payload": "{\"risk_score\": 0.12, \"payment_verified\": true}",
    },
    {
        "event_id": "EVT002",
        "event_time": "2026-05-01T10:20:00Z",
        "source_system": "web_store",
        "customer": {
            "customer_id": "C002",
            "name": "Bob Smith",
            "segment": "silver",
        },
        "device": {
            "platform": "web",
            "app_version": "browser",
            "device_id": "WEB-778",
        },
        "location": {
            "country": "TH",
            "city": "Chiang Mai",
        },
        "items": [
            {"product_id": "P003", "product_name": "Notebook Stand", "quantity": 1, "price": 1250.0},
        ],
        "attributes": {
            "campaign": "organic",
            "channel": "web",
        },
        "event_payload": "{\"risk_score\": 0.35, \"payment_verified\": true}",
    },
    {
        "event_id": "EVT003",
        "event_time": "2026-05-02T08:05:00Z",
        "source_system": "mobile_app",
        "customer": {
            "customer_id": "C003",
            "name": "Chai Lee",
            "segment": "new",
        },
        "device": {
            "platform": "android",
            "app_version": "2.3.9",
            "device_id": "AND-221",
        },
        "location": {
            "country": "TH",
            "city": "Rayong",
        },
        "items": [
            {"product_id": "P004", "product_name": "Keyboard", "quantity": 1, "price": 1590.0},
            {"product_id": "P005", "product_name": "Mouse Pad", "quantity": 1, "price": 390.0},
            {"product_id": "P002", "product_name": "USB-C Cable", "quantity": 1, "price": 250.0},
        ],
        "attributes": {
            "campaign": "android_sale",
            "channel": "in_app_banner",
        },
        "event_payload": "{\"risk_score\": 0.82, \"payment_verified\": false}",
    },
    {
        "event_id": "EVT004",
        "event_time": "2026-05-02T11:30:00Z",
        "source_system": "partner_api",
        "customer": {
            "customer_id": "C004",
            "name": "Dana Kim",
            "segment": None,
        },
        "device": {
            "platform": "api",
            "app_version": None,
            "device_id": None,
        },
        "location": {
            "country": "TH",
            "city": "Phuket",
        },
        "items": [],
        "attributes": {
            "campaign": "partner_referral",
            "channel": "api",
        },
        "event_payload": "{\"risk_score\": 0.55, \"payment_verified\": true}",
    },
]

# Convert each Python dict into one JSON string line for JSON Lines / NDJSON format
api_event_json_lines = [json.dumps(row) for row in api_events]

# Tips:
# The "s" in dumps/loads means string.

# String-based JSON helpers
# json.dumps(): Python dict / list of dict -> JSON string
# json.loads(): JSON string -> Python dict / list of dict
# Use dumps/loads when working with JSON as string, such as API payload or JSON Lines.

# File-based JSON helpers
# json.dump(): Python dict / list of dict -> JSON file
# json.load(): JSON file -> Python dict / list of dict
# Use dump/load when reading or writing small JSON files with Python file objects, such as open().

# In PySpark, prefer spark.read.json(path) to read JSON files as a distributed DataFrame.

print("Sample JSON line:")
print(api_event_json_lines[0])

StatementMeta(, b565ef77-0fec-4050-a2b7-11d6c8839678, 4, Finished, Available, Finished, False)

Sample JSON line:
{"event_id": "EVT001", "event_time": "2026-05-01T09:15:00Z", "source_system": "mobile_app", "customer": {"customer_id": "C001", "name": "Alice Wong", "segment": "gold"}, "device": {"platform": "ios", "app_version": "2.4.1", "device_id": "IOS-001"}, "location": {"country": "TH", "city": "Bangkok"}, "items": [{"product_id": "P001", "product_name": "Wireless Mouse", "quantity": 1, "price": 890.0}, {"product_id": "P002", "product_name": "USB-C Cable", "quantity": 2, "price": 250.0}], "attributes": {"campaign": "may_promo", "channel": "push_notification", "coupon": "MAY100"}, "event_payload": "{\"risk_score\": 0.12, \"payment_verified\": true}"}


### Define explicit nested schema

สำหรับ nested JSON ควรใช้ explicit schema เพราะ schema inference อาจทำให้ type ไม่คงที่เมื่อ source ส่งข้อมูลเปลี่ยน เช่นบาง batch ไม่มี `items`, บาง batch มี key ใน `attributes` ไม่ครบ หรือบาง field กลายเป็น string ทั้งหมด


In [3]:
item_schema = T.StructType([
    T.StructField("product_id", T.StringType(), True),
    T.StructField("product_name", T.StringType(), True),
    T.StructField("quantity", T.IntegerType(), True),
    T.StructField("price", T.DoubleType(), True),
])

# Define schema for parsing event_payload from JSON string into a nested struct
payload_schema = T.StructType([
    T.StructField("risk_score", T.DoubleType(), True),
    T.StructField("payment_verified", T.BooleanType(), True),
])

event_schema = T.StructType([
    T.StructField("event_id", T.StringType(), False),
    T.StructField("event_time", T.StringType(), True),
    T.StructField("source_system", T.StringType(), True),
    T.StructField("customer", T.StructType([
        T.StructField("customer_id", T.StringType(), True),
        T.StructField("name", T.StringType(), True),
        T.StructField("segment", T.StringType(), True),
    ]), True),
    T.StructField("device", T.StructType([
        T.StructField("platform", T.StringType(), True),
        T.StructField("app_version", T.StringType(), True),
        T.StructField("device_id", T.StringType(), True),
    ]), True),
    T.StructField("location", T.StructType([
        T.StructField("country", T.StringType(), True),
        T.StructField("city", T.StringType(), True),
    ]), True),
    T.StructField("items", T.ArrayType(item_schema), True),
    T.StructField("attributes", T.MapType(T.StringType(), T.StringType()), True),
    T.StructField("event_payload", T.StringType(), True),
])

# Tip:
# MapType(StringType, StringType) means key-value pairs where both keys and values are strings

StatementMeta(, b565ef77-0fec-4050-a2b7-11d6c8839678, 5, Finished, Available, Finished, False)

### Write and read JSON sample

Code cell นี้พยายามเขียน sample JSON ลง Lakehouse `Files/day09_nested_json_complex_types/api_events.json` ก่อน แล้วอ่านกลับด้วย `spark.read.schema(...).json(...)`


In [4]:
json_lakehouse_dir = "Files/day09_nested_json_complex_types"
json_lakehouse_path = f"{json_lakehouse_dir}/api_events.jsonl"

mssparkutils.fs.mkdirs(json_lakehouse_dir)
mssparkutils.fs.put(
    json_lakehouse_path,              # Target .jsonl file path in Fabric Lakehouse Files
    "\n".join(api_event_json_lines),  # Join JSON strings with newline (\n) for JSON Lines / NDJSON format
    True                              # Overwrite the file if it already exists
)
print(f"Wrote JSON sample to Lakehouse Files: {json_lakehouse_path}")

df_events_raw = spark.read.schema(event_schema).json(json_lakehouse_path)

df_events_raw.printSchema()
df_events_raw.show(truncate=False)

StatementMeta(, b565ef77-0fec-4050-a2b7-11d6c8839678, 6, Finished, Available, Finished, False)

Wrote JSON sample to Lakehouse Files: Files/day09_nested_json_complex_types/api_events.jsonl
root
 |-- event_id: string (nullable = true)
 |-- event_time: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- customer: struct (nullable = true)
 |    |-- customer_id: string (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- segment: string (nullable = true)
 |-- device: struct (nullable = true)
 |    |-- platform: string (nullable = true)
 |    |-- app_version: string (nullable = true)
 |    |-- device_id: string (nullable = true)
 |-- location: struct (nullable = true)
 |    |-- country: string (nullable = true)
 |    |-- city: string (nullable = true)
 |-- items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- product_id: string (nullable = true)
 |    |    |-- product_name: string (nullable = true)
 |    |    |-- quantity: integer (nullable = true)
 |    |    |-- price: double (nullable = true)
 |-- attribu

## PySpark code examples

ใน section นี้จะไล่จากการอ่าน schema → access nested fields → extract map → parse JSON string → explode array → flatten เป็น item-level table แล้วเขียนลง Lakehouse table ขนาดเล็ก


### Example 1: Inspect nested schema

`printSchema()` สำคัญมากสำหรับ nested JSON เพราะช่วยให้เห็นว่า column ไหนเป็น `struct`, `array`, หรือ `map` และต้องเข้าถึง field ชั้นในยังไง


In [5]:
df_events_raw.printSchema()

print("Raw event count:", df_events_raw.count())

StatementMeta(, b565ef77-0fec-4050-a2b7-11d6c8839678, 7, Finished, Available, Finished, False)

root
 |-- event_id: string (nullable = true)
 |-- event_time: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- customer: struct (nullable = true)
 |    |-- customer_id: string (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- segment: string (nullable = true)
 |-- device: struct (nullable = true)
 |    |-- platform: string (nullable = true)
 |    |-- app_version: string (nullable = true)
 |    |-- device_id: string (nullable = true)
 |-- location: struct (nullable = true)
 |    |-- country: string (nullable = true)
 |    |-- city: string (nullable = true)
 |-- items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- product_id: string (nullable = true)
 |    |    |-- product_name: string (nullable = true)
 |    |    |-- quantity: integer (nullable = true)
 |    |    |-- price: double (nullable = true)
 |-- attributes: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull =

### Example 2: Access struct fields

ตัวอย่างนี้ flatten field จาก `customer`, `device` และ `location` ออกมาเป็น event-level columns โดยยังคง grain เป็น 1 row ต่อ 1 event เพราะ field เหล่านี้เป็น nested object เดียว (`StructType`) ไม่ใช่ array


In [6]:
df_event_flat_base = (
    df_events_raw
    .select(
        "event_id",
        F.to_timestamp("event_time", "yyyy-MM-dd'T'HH:mm:ss'Z'").alias("event_timestamp"),
        "source_system",
        F.col("customer.customer_id").alias("customer_id"),
        F.col("customer.name").alias("customer_name"),
        F.coalesce(F.col("customer.segment"), F.lit("unknown")).alias("customer_segment"),
        F.col("device.platform").alias("device_platform"),
        F.col("device.app_version").alias("app_version"),
        F.col("location.country").alias("country"),
        F.col("location.city").alias("city"),
        F.size("items").alias("item_count"),
    )
)

# Tip:
# - size(): returns the number of elements in an array or key-value pairs in a map

df_event_flat_base.show(truncate=False)
df_event_flat_base.printSchema()

StatementMeta(, b565ef77-0fec-4050-a2b7-11d6c8839678, 8, Finished, Available, Finished, False)

+--------+-------------------+-------------+-----------+-------------+----------------+---------------+-----------+-------+----------+----------+
|event_id|event_timestamp    |source_system|customer_id|customer_name|customer_segment|device_platform|app_version|country|city      |item_count|
+--------+-------------------+-------------+-----------+-------------+----------------+---------------+-----------+-------+----------+----------+
|EVT001  |2026-05-01 09:15:00|mobile_app   |C001       |Alice Wong   |gold            |ios            |2.4.1      |TH     |Bangkok   |2         |
|EVT002  |2026-05-01 10:20:00|web_store    |C002       |Bob Smith    |silver          |web            |browser    |TH     |Chiang Mai|1         |
|EVT003  |2026-05-02 08:05:00|mobile_app   |C003       |Chai Lee     |new             |android        |2.3.9      |TH     |Rayong    |3         |
|EVT004  |2026-05-02 11:30:00|partner_api  |C004       |Dana Kim     |unknown         |api            |NULL       |TH     |P

### Example 3: Extract values from MapType

`attributes` เป็น `MapType` เพราะ key อาจยืดหยุ่น เช่นบาง event มี `coupon` บาง event ไม่มี

ใช้ `F.element_at()` เพื่อดึง value จาก key ที่ต้องการออกมาเป็น column


In [7]:
df_event_attributes = (
    df_events_raw
    .select(
        "event_id",
        F.map_keys("attributes").alias("attribute_keys"),
        F.element_at("attributes", "campaign").alias("campaign"),
        F.element_at("attributes", "channel").alias("channel"),
        F.element_at("attributes", "coupon").alias("coupon"),
    )
)

# Tips:
# - map_keys(): returns keys from a MapType column
# - element_at(): extracts value by key; missing keys return null

df_event_attributes.show(truncate=False)

StatementMeta(, b565ef77-0fec-4050-a2b7-11d6c8839678, 9, Finished, Available, Finished, False)

+--------+---------------------------+----------------+-----------------+------+
|event_id|attribute_keys             |campaign        |channel          |coupon|
+--------+---------------------------+----------------+-----------------+------+
|EVT001  |[campaign, channel, coupon]|may_promo       |push_notification|MAY100|
|EVT002  |[campaign, channel]        |organic         |web              |NULL  |
|EVT003  |[campaign, channel]        |android_sale    |in_app_banner    |NULL  |
|EVT004  |[campaign, channel]        |partner_referral|api              |NULL  |
+--------+---------------------------+----------------+-----------------+------+



### Example 4: Parse JSON string column with from_json

`event_payload` เป็น string แต่ข้างในเป็น JSON อีกชั้น จึงต้องใช้ `from_json` เพื่อ parse ให้กลายเป็น struct ก่อน extract field ย่อย


In [8]:
# Parse JSON string column into a StructType column
df_payload_parsed = (
    df_events_raw
    .withColumn("payload_parsed", F.from_json("event_payload", payload_schema))
)

# Compare raw JSON string vs parsed struct
df_payload_parsed.select("event_id", "event_payload", "payload_parsed").show(truncate=False)
df_payload_parsed.select("event_id", "event_payload", "payload_parsed").printSchema()

# Flatten parsed struct fields into normal columns
df_event_payload = (
    df_payload_parsed
    .select(
        "event_id",
        F.col("payload_parsed.risk_score").alias("risk_score"),
        F.col("payload_parsed.payment_verified").alias("payment_verified"),
    )
)

df_event_payload.show(truncate=False)
df_event_payload.printSchema()

StatementMeta(, b565ef77-0fec-4050-a2b7-11d6c8839678, 10, Finished, Available, Finished, False)

+--------+-----------------------------------------------+--------------+
|event_id|event_payload                                  |payload_parsed|
+--------+-----------------------------------------------+--------------+
|EVT001  |{"risk_score": 0.12, "payment_verified": true} |{0.12, true}  |
|EVT002  |{"risk_score": 0.35, "payment_verified": true} |{0.35, true}  |
|EVT003  |{"risk_score": 0.82, "payment_verified": false}|{0.82, false} |
|EVT004  |{"risk_score": 0.55, "payment_verified": true} |{0.55, true}  |
+--------+-----------------------------------------------+--------------+

root
 |-- event_id: string (nullable = true)
 |-- event_payload: string (nullable = true)
 |-- payload_parsed: struct (nullable = true)
 |    |-- risk_score: double (nullable = true)
 |    |-- payment_verified: boolean (nullable = true)

+--------+----------+----------------+
|event_id|risk_score|payment_verified|
+--------+----------+----------------+
|EVT001  |0.12      |true            |
|EVT002  |0.3

### Example 5: Explode array into item-level rows

`items` เป็น array ของ struct ถ้าต้องการวิเคราะห์สินค้าในแต่ละ event ต้อง explode ออกมาเป็นหลาย rows

ในตัวอย่างนี้ใช้ `explode_outer` เพื่อให้ event ที่ไม่มี item เช่น `EVT004` ยังไม่หายไปทันทีตอนตรวจข้อมูล


In [9]:
df_event_items = (
    df_events_raw
    .select(
        "event_id",
        F.col("customer.customer_id").alias("customer_id"),
        F.explode_outer("items").alias("item"),
    )
    .select(
        "event_id",
        "customer_id",
        F.col("item.product_id").alias("product_id"),
        F.col("item.product_name").alias("product_name"),
        F.col("item.quantity").alias("quantity"),
        F.col("item.price").alias("price"),
    )
    .withColumn("line_amount", F.col("quantity") * F.col("price"))
)

# Tips:
# - First select: explode items so each item becomes one row 
# - Second select: flatten item fields into normal columns

df_event_items.show(truncate=False)
print("Item-level row count:", df_event_items.count())

StatementMeta(, b565ef77-0fec-4050-a2b7-11d6c8839678, 11, Finished, Available, Finished, False)

+--------+-----------+----------+--------------+--------+------+-----------+
|event_id|customer_id|product_id|product_name  |quantity|price |line_amount|
+--------+-----------+----------+--------------+--------+------+-----------+
|EVT001  |C001       |P001      |Wireless Mouse|1       |890.0 |890.0      |
|EVT001  |C001       |P002      |USB-C Cable   |2       |250.0 |500.0      |
|EVT002  |C002       |P003      |Notebook Stand|1       |1250.0|1250.0     |
|EVT003  |C003       |P004      |Keyboard      |1       |1590.0|1590.0     |
|EVT003  |C003       |P005      |Mouse Pad     |1       |390.0 |390.0      |
|EVT003  |C003       |P002      |USB-C Cable   |1       |250.0 |250.0      |
|EVT004  |C004       |NULL      |NULL          |NULL    |NULL  |NULL       |
+--------+-----------+----------+--------------+--------+------+-----------+

Item-level row count: 7


### Example 6: Build flattened item-level table

ตัวอย่างนี้รวม struct fields, map attributes, parsed payload และ exploded items ให้กลายเป็น item-level table ที่พร้อมใช้ต่อ

ข้อควรสังเกต: output table นี้เป็น **item-level grain** ไม่ใช่ event-level grain แล้ว


In [10]:
df_event_items_flattened = (
    df_events_raw
    .withColumn("payload", F.from_json("event_payload", payload_schema))
    .withColumn("item", F.explode_outer("items"))
    .select(
        "event_id",
        F.to_timestamp("event_time", "yyyy-MM-dd'T'HH:mm:ss'Z'").alias("event_timestamp"),
        "source_system",
        F.col("customer.customer_id").alias("customer_id"),
        F.col("customer.name").alias("customer_name"),
        F.coalesce(F.col("customer.segment"), F.lit("unknown")).alias("customer_segment"),
        F.col("device.platform").alias("device_platform"),
        F.col("location.city").alias("city"),
        F.element_at("attributes", "campaign").alias("campaign"),
        F.element_at("attributes", "channel").alias("channel"),
        F.col("payload.risk_score").alias("risk_score"),
        F.col("payload.payment_verified").alias("payment_verified"),
        F.col("item.product_id").alias("product_id"),
        F.col("item.product_name").alias("product_name"),
        F.col("item.quantity").alias("quantity"),
        F.col("item.price").alias("price"),
    )
    .withColumn("line_amount", F.col("quantity") * F.col("price"))
)

df_event_items_flattened.show(truncate=False)
df_event_items_flattened.printSchema()

StatementMeta(, b565ef77-0fec-4050-a2b7-11d6c8839678, 12, Finished, Available, Finished, False)

+--------+-------------------+-------------+-----------+-------------+----------------+---------------+----------+----------------+-----------------+----------+----------------+----------+--------------+--------+------+-----------+
|event_id|event_timestamp    |source_system|customer_id|customer_name|customer_segment|device_platform|city      |campaign        |channel          |risk_score|payment_verified|product_id|product_name  |quantity|price |line_amount|
+--------+-------------------+-------------+-----------+-------------+----------------+---------------+----------+----------------+-----------------+----------+----------------+----------+--------------+--------+------+-----------+
|EVT001  |2026-05-01 09:15:00|mobile_app   |C001       |Alice Wong   |gold            |ios            |Bangkok   |may_promo       |push_notification|0.12      |true            |P001      |Wireless Mouse|1       |890.0 |890.0      |
|EVT001  |2026-05-01 09:15:00|mobile_app   |C001       |Alice Wong   |go

### Example 7: Create event-level summary after explode

หลัง explode แล้วควรตรวจ row count และ aggregate กลับเพื่อเช็กว่าตัวเลขสอดคล้องกับ event เดิมหรือไม่


In [11]:
df_event_amount_summary = (
    df_event_items_flattened
    .where(F.col("product_id").isNotNull())
    .groupBy("event_id", "customer_id")
    .agg(
        F.count("product_id").alias("product_count"),
        F.sum("quantity").alias("total_quantity"),
        F.round(F.sum("line_amount"), 2).alias("total_amount"),
    )
)

df_event_amount_summary.show(truncate=False)

StatementMeta(, b565ef77-0fec-4050-a2b7-11d6c8839678, 13, Finished, Available, Finished, False)

+--------+-----------+-------------+--------------+------------+
|event_id|customer_id|product_count|total_quantity|total_amount|
+--------+-----------+-------------+--------------+------------+
|EVT003  |C003       |3            |3             |2230.0      |
|EVT002  |C002       |1            |1             |1250.0      |
|EVT001  |C001       |2            |3             |1390.0      |
+--------+-----------+-------------+--------------+------------+



### Example 8: Write flattened result to Lakehouse table

การ write table เป็น action และใน Fabric ควร attach Lakehouse ก่อน run cell นี้


In [12]:
output_table = "day09_event_items_flattened"

(
    df_event_items_flattened
    .where(F.col("product_id").isNotNull())
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(output_table)
)

spark.read.table(output_table).show(truncate=False)

StatementMeta(, b565ef77-0fec-4050-a2b7-11d6c8839678, 14, Finished, Available, Finished, False)

+--------+-------------------+-------------+-----------+-------------+----------------+---------------+----------+------------+-----------------+----------+----------------+----------+--------------+--------+------+-----------+
|event_id|event_timestamp    |source_system|customer_id|customer_name|customer_segment|device_platform|city      |campaign    |channel          |risk_score|payment_verified|product_id|product_name  |quantity|price |line_amount|
+--------+-------------------+-------------+-----------+-------------+----------------+---------------+----------+------------+-----------------+----------+----------------+----------+--------------+--------+------+-----------+
|EVT001  |2026-05-01 09:15:00|mobile_app   |C001       |Alice Wong   |gold            |ios            |Bangkok   |may_promo   |push_notification|0.12      |true            |P001      |Wireless Mouse|1       |890.0 |890.0      |
|EVT001  |2026-05-01 09:15:00|mobile_app   |C001       |Alice Wong   |gold            |i

## Quick recap

| Question | Answer |
|---|---|
| `StructType` ใช้แทนอะไร? | Nested object ที่มี field ย่อย เช่น `customer.customer_id` |
| `ArrayType` ใช้แทนอะไร? | List ของค่า หรือ list ของ nested object เช่น `items` ที่เก็บหลาย item ในหนึ่ง event |
| `MapType` เหมาะกับข้อมูลแบบไหน? | Dynamic key-value attributes เช่น `attributes["campaign"]`, `attributes["channel"]` |
| ใช้ function อะไรแตก array เป็นหลาย rows? | `explode` หรือ `explode_outer` |
| ทำไมต้องระวังหลัง `explode`? | Grain และ row count ของ DataFrame เปลี่ยนทันที |
| ใช้ function อะไร parse JSON string column? | `from_json` พร้อม explicit schema |


## Exercises


### Exercise 1: Create event-level flattened DataFrame

สร้าง DataFrame ชื่อ `df_ex1_event_level` โดย flatten เฉพาะ event-level fields ไม่ต้อง explode items

Requirements:

- ดึง `event_id`, `event_timestamp`, `source_system`
- ดึง `customer_id`, `customer_name`, `customer_segment`
- ดึง `device_platform`, `city`
- เพิ่ม `item_count` จากจำนวน element ใน `items`

Expected result:

- จำนวน rows ต้องเท่ากับจำนวน raw events
- ยังไม่เกิด item-level duplication


In [13]:
df_ex1_event_level = (
    df_events_raw
    .select(
        "event_id",
        F.to_timestamp("event_time", "yyyy-MM-dd'T'HH:mm:ss'Z'").alias("event_timestamp"),
        "source_system",
        F.col("customer.customer_id").alias("customer_id"),
        F.col("customer.name").alias("customer_name"),
        F.coalesce(F.col("customer.segment"), F.lit("unknown")).alias("customer_segment"),
        F.col("device.platform").alias("device_platform"),
        F.col("location.city").alias("city"),
        F.size("items").alias("item_count"),
    )
)

df_ex1_event_level.show(truncate=False)
print("Raw events:", df_events_raw.count())
print("Event-level rows:", df_ex1_event_level.count())

StatementMeta(, b565ef77-0fec-4050-a2b7-11d6c8839678, 15, Finished, Available, Finished, False)

+--------+-------------------+-------------+-----------+-------------+----------------+---------------+----------+----------+
|event_id|event_timestamp    |source_system|customer_id|customer_name|customer_segment|device_platform|city      |item_count|
+--------+-------------------+-------------+-----------+-------------+----------------+---------------+----------+----------+
|EVT001  |2026-05-01 09:15:00|mobile_app   |C001       |Alice Wong   |gold            |ios            |Bangkok   |2         |
|EVT002  |2026-05-01 10:20:00|web_store    |C002       |Bob Smith    |silver          |web            |Chiang Mai|1         |
|EVT003  |2026-05-02 08:05:00|mobile_app   |C003       |Chai Lee     |new             |android        |Rayong    |3         |
|EVT004  |2026-05-02 11:30:00|partner_api  |C004       |Dana Kim     |unknown         |api            |Phuket    |0         |
+--------+-------------------+-------------+-----------+-------------+----------------+---------------+----------+----

### Exercise 2: Create item-level DataFrame and calculate line amount

สร้าง DataFrame ชื่อ `df_ex2_item_level` โดยแตก `items` ออกมาเป็น item-level rows และคำนวณ `line_amount`

Requirements:

- ใช้ `explode_outer("items")`
- ดึง `product_id`, `product_name`, `quantity`, `price`
- สร้าง `line_amount = quantity * price`
- แสดงเฉพาะ records ที่ `product_id` ไม่เป็น null เมื่อต้องการ item จริง

Expected result:

- Event ที่มีหลาย items จะกลายเป็นหลาย rows
- `line_amount` ต้องคำนวณได้ถูกต้องต่อ item


In [14]:
df_ex2_item_level = (
    df_events_raw
    .select(
        "event_id",
        F.col("customer.customer_id").alias("customer_id"),
        F.explode_outer("items").alias("item"),
    )
    .select(
        "event_id",
        "customer_id",
        F.col("item.product_id").alias("product_id"),
        F.col("item.product_name").alias("product_name"),
        F.col("item.quantity").alias("quantity"),
        F.col("item.price").alias("price"),
    )
    .withColumn("line_amount", F.col("quantity") * F.col("price"))
)

df_ex2_item_level.show(truncate=False)

print("Rows after explode_outer:", df_ex2_item_level.count())
print("Actual item rows:", df_ex2_item_level.where(F.col("product_id").isNotNull()).count())

StatementMeta(, b565ef77-0fec-4050-a2b7-11d6c8839678, 16, Finished, Available, Finished, False)

+--------+-----------+----------+--------------+--------+------+-----------+
|event_id|customer_id|product_id|product_name  |quantity|price |line_amount|
+--------+-----------+----------+--------------+--------+------+-----------+
|EVT001  |C001       |P001      |Wireless Mouse|1       |890.0 |890.0      |
|EVT001  |C001       |P002      |USB-C Cable   |2       |250.0 |500.0      |
|EVT002  |C002       |P003      |Notebook Stand|1       |1250.0|1250.0     |
|EVT003  |C003       |P004      |Keyboard      |1       |1590.0|1590.0     |
|EVT003  |C003       |P005      |Mouse Pad     |1       |390.0 |390.0      |
|EVT003  |C003       |P002      |USB-C Cable   |1       |250.0 |250.0      |
|EVT004  |C004       |NULL      |NULL          |NULL    |NULL  |NULL       |
+--------+-----------+----------+--------------+--------+------+-----------+

Rows after explode_outer: 7
Actual item rows: 6


### Exercise 3: Parse payload and create risk flag

สร้าง DataFrame ชื่อ `df_ex3_risk_events` โดย parse `event_payload` แล้วสร้าง `risk_level`

Requirements:

- ใช้ `from_json(event_payload, payload_schema)`
- ดึง `risk_score` และ `payment_verified`
- สร้าง `risk_level`:
  - `high` ถ้า `risk_score >= 0.80`
  - `medium` ถ้า `risk_score >= 0.50`
  - ที่เหลือเป็น `low`

Expected result:

- Event `EVT003` ควรเป็น `high`
- Event ที่เหลือควรอยู่ใน `low` หรือ `medium` ตาม threshold


In [15]:
df_ex3_risk_events = (
    df_events_raw
    .withColumn("payload", F.from_json("event_payload", payload_schema))
    .select(
        "event_id",
        F.col("customer.customer_id").alias("customer_id"),
        F.col("payload.risk_score").alias("risk_score"),
        F.col("payload.payment_verified").alias("payment_verified"),
    )
    .withColumn(
        "risk_level",
        F.when(F.col("risk_score") >= 0.80, F.lit("high"))
         .when(F.col("risk_score") >= 0.50, F.lit("medium"))
         .otherwise(F.lit("low"))
    )
)

df_ex3_risk_events.show(truncate=False)

StatementMeta(, b565ef77-0fec-4050-a2b7-11d6c8839678, 17, Finished, Available, Finished, False)

+--------+-----------+----------+----------------+----------+
|event_id|customer_id|risk_score|payment_verified|risk_level|
+--------+-----------+----------+----------------+----------+
|EVT001  |C001       |0.12      |true            |low       |
|EVT002  |C002       |0.35      |true            |low       |
|EVT003  |C003       |0.82      |false           |high      |
|EVT004  |C004       |0.55      |true            |medium    |
+--------+-----------+----------+----------------+----------+



## Common mistakes

1. **ใช้ schema inference กับ nested JSON โดยไม่ตรวจ schema**

   Nested field บางตัวอาจถูก infer ผิด หรือหายเป็น null เมื่อ batch ใหม่มี structure ไม่เหมือนเดิม ควรใช้ explicit schema เมื่อเป็น pipeline ที่ต้องรันซ้ำ

2. **ลืมว่า `explode` เปลี่ยน grain ของข้อมูล**

   จาก event-level อาจกลายเป็น item-level ทันที ถ้าเอา row count ไปเทียบกับ source โดยไม่เข้าใจ grain จะสรุปผิดได้

3. **ใช้ `explode` แล้ว records ที่ array ว่างหายไปโดยไม่รู้ตัว**

   ถ้าต้องการเก็บ row ต้นทาง เช่น event ที่ `items` เป็น array ว่าง ไว้เพื่อตรวจข้อมูล ให้ใช้ `explode_outer` ก่อน แล้วค่อย filter เฉพาะ item ที่มีค่าจริงภายหลัง

4. **Drop raw nested column เร็วเกินไป**

   ถ้าเกิด source issue ภายหลัง อาจ debug ยากว่า raw payload เดิมหน้าตาเป็นอย่างไร ควรเก็บ raw/bronze version ไว้หรืออย่างน้อยไม่ลบทิ้งเร็วเกินไปในขั้นตอนตรวจสอบ

5. **Flatten ทุกอย่างโดยไม่มี naming convention**

   Column เช่น `id`, `name`, `status` อาจชนกันระหว่าง customer, product, device ควรตั้งชื่อให้ชัด เช่น `customer_id`, `product_id`, `device_platform`

6. **Assume ว่า map key มีครบทุก record**

   `attributes["coupon"]` หรือ `element_at(attributes, "coupon")` อาจได้ null ถ้า key ไม่มี ต้องออกแบบ downstream ให้รับ null ได้


## Expected Output / Success Criteria

เมื่อจบ Day 09 ควรทำได้:

- อธิบายความต่างระหว่าง `StructType`, `ArrayType` และ `MapType` ได้
- อ่านไฟล์ nested JSON โดยกำหนด explicit schema เองล่วงหน้าได้
- ใช้ `printSchema()` เพื่อเข้าใจ nested structure ก่อน transform ได้
- Access nested fields เช่น `customer.customer_id`, `device.platform`, `location.city` ได้
- Extract map keys ด้วย `element_at`, `map_keys` หรือ related map functions ได้
- Parse JSON string column ด้วย `from_json` ได้
- ใช้ `explode` / `explode_outer` เพื่อแตก array เป็นหลาย rows ได้
- อธิบายได้ว่า `explode` เปลี่ยน grain และ row count ของ DataFrame อย่างไร
- Flatten nested JSON เป็น event-level และ item-level DataFrame ได้
- Write flattened result เป็น Lakehouse table และอ่านกลับด้วย `spark.read.table()` ได้ ถ้า attach Lakehouse เรียบร้อย


## Final takeaway

Nested JSON ไม่ได้ยากเพราะ syntax อย่างเดียว แต่ยากเพราะต้องรู้ว่า data structure และ grain เปลี่ยนอย่างไรระหว่างทาง

> Read schema first, decide the grain, then flatten intentionally.

Mindset ที่ควรจำจาก Day 09 คือ:

- `struct` คือ nested object ที่ access field ย่อยได้
- `array` ต้องระวังเป็นพิเศษ เพราะ `explode` ทำให้จำนวน rows เปลี่ยน
- `map` เหมาะกับ dynamic attributes แต่ key อาจไม่มีครบทุก record
- `from_json` ใช้เมื่อข้อมูลใน column เป็น string ที่ข้างในเป็น JSON อีกชั้น
- ก่อน write output ต้องรู้ว่า table นี้เป็น event-level หรือ item-level


## Optional cleanup


In [ ]:
# spark.sql("DROP TABLE IF EXISTS day09_event_items_flattened")